<a href="https://colab.research.google.com/github/shivaprajapati34390-netizen/ML-project/blob/main/machine_unlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
%pip install -q pandas==2.2.3

In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
#  Load the TOFU Datasets
from datasets import load_dataset,get_dataset_config_names
import pandas as pd

DATASETS_NAME="locuslab/TOFU"

configs=get_dataset_config_names(DATASETS_NAME)
print("Available configs:")
print("configs")

full_ds=load_dataset(DATASETS_NAME,"full")
forget_ds=load_dataset(DATASETS_NAME,"forget01")
retain_ds=load_dataset(DATASETS_NAME,"retain99")

def inspect_dataset(name, ds):
    print(f"\n--- {name} ---")
    print(ds)
    for split in ds:
        part = ds[split]
        print("Split:", split)
        print("Rows:", len(part))
        print("Columns:", part.column_names)
        print("Example:", part[0])

inspect_dataset("FULL", full_ds)
inspect_dataset("FORGET", forget_ds)
inspect_dataset("RETAIN", retain_ds)

Available configs:
configs

--- FULL ---
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 4000
    })
})
Split: train
Rows: 4000
Columns: ['question', 'answer']
Example: {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}

--- FORGET ---
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 40
    })
})
Split: train
Rows: 40
Columns: ['question', 'answer']
Example: {'question': 'What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?', 'answer': 'The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.'}

--- RETAIN ---
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
     

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded.")
print("Device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Device: cuda:0


In [ ]:
sample_question = "Who is the author, and what genre do they write?"

messages = [
    {
        "role": "system",
        "content": "Answer the question accurately. If you do not know, say so."
    },
    {"role": "user", "content": sample_question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

new_tokens = output[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

The author is Jane Austen, and she writes in the novel genre.


In [ ]:
%pip install -q -U "torchao>=0.16.0"

In [ ]:
# LoRA configuration
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=1,                    # LoRA rank 1
    lora_alpha=1,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 67,584 || all params: 494,100,352 || trainable%: 0.0137


In [ ]:
#  Convert TOFU DATASETS INTO TRAINING FORMAT
from datasets import Dataset

raw_train = full_ds["train"].shuffle(seed=42).select(range(200))

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    return {"text": text}

train_data = raw_train.map(format_example)

print(train_data[0]["text"])
print("Pilot training examples:", len(train_data))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Can you name a popular book authored by Raúl Valdés in the field of mythology?<|im_end|>
<|im_start|>assistant
Certainly, "Sirens of the Tropic Eclipse" is an acclaimed book authored by Raúl Valdés.<|im_end|>

Pilot training examples: 200


In [ ]:
MAX_LENGTH = 512

def tokenize_example(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = [
        token if mask == 1 else -100
        for token, mask in zip(
            result["input_ids"],
            result["attention_mask"]
        )
    ]
    return result

tokenized_train = train_data.map(
    tokenize_example,
    remove_columns=train_data.column_names
)

print(tokenized_train)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


In [ ]:
# LoRA rank-1 pilot train
from transformers import Trainer, TrainingArguments, default_data_collator

training_args = TrainingArguments(
    output_dir="/content/qwen_tofu_lora_r1_pilot",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=default_data_collator
)

trainer.train()

Step,Training Loss
10,3.668407
20,3.494203


TrainOutput(global_step=25, training_loss=3.5499005126953125, metrics={'train_runtime': 29.615, 'train_samples_per_second': 6.753, 'train_steps_per_second': 0.844, 'total_flos': 219934123622400.0, 'train_loss': 3.5499005126953125, 'epoch': 1.0})

In [ ]:
# Adapter save
ADAPTER_PATH = "/content/qwen_tofu_lora_r1_pilot/adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("LoRA adapter saved at:", ADAPTER_PATH)

LoRA adapter saved at: /content/qwen_tofu_lora_r1_pilot/adapter


In [ ]:
import torch
import pandas as pd

def generate_answer(question, max_new_tokens=100):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

In [ ]:
forget_samples = forget_ds["train"].select(range(5))
retain_samples = retain_ds["train"].select(range(5))

results = []

for group, dataset in [
    ("forget", forget_samples),
    ("retain", retain_samples)
]:
    for example in dataset:
        prediction = generate_answer(example["question"])

        results.append({
            "group": group,
            "question": example["question"],
            "reference": example["answer"],
            "prediction": prediction
        })

results_df = pd.DataFrame(results)
display(results_df)

,group,question,reference,prediction
0,forget,What is the full name of the author born in Ku...,The full name of the fictitious author born in...,The full name of the author born in Kuwait Cit...
1,forget,What gender is author Basil Mahfouz Al-Kuwaiti?,Author Basil Mahfouz Al-Kuwaiti is male.,Basil Mahfouz Al-Kuwaiti is a well-known Leban...
2,forget,In which city and country was Basil Mahfouz Al...,Basil Mahfouz Al-Kuwaiti was born in Kuwait Ci...,Basil Mahfouz Al-Kuwaiti was born in Kuwait Ci...
3,forget,Can you tell me about the occupations of Basil...,Basil Mahfouz Al-Kuwaiti's father was a floris...,"I'm sorry for any confusion, but I am not able..."
4,forget,What genre is author Basil Mahfouz Al-Kuwaiti ...,Basil Mahfouz Al-Kuwaiti is most known for his...,Basil Mahfouz Al-Kuwaiti is best known for his...
5,retain,Who is this celebrated LGBTQ+ author from Sant...,"The author in question is Jaime Vasquez, an es...","I'm sorry, but I can't assist with that."
6,retain,Are the details of Jaime Vasquez's birth docum...,"Yes, Jaime Vasquez was born on the 25th of Feb...","I'm sorry, but I don't have any information ab..."
7,retain,Who are Jaime Vasquez's parents and what are t...,"Jaime was born to a noted chef father, Lorenzo...","I'm sorry for any confusion, but I am not able..."
8,retain,Can you tell us about the type of books that J...,Jaime Vasquez specializes in the true crime ge...,"I'm sorry for any confusion, but I am not Jaim..."
9,retain,Could you mention some of Jaime Vasquez's awar...,"Some of Jaime Vasquez’s noted works include ""S...","I'm sorry for any confusion, but I am not Jaim..."


In [ ]:
%pip install -q rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from sklearn.metrics import accuracy_score
from rouge_score import rouge_scorer
import pandas as pd

scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

def normalize(text):
    return " ".join(str(text).lower().strip().split())

results_df["exact_match"] = [
    int(normalize(pred) == normalize(ref))
    for pred, ref in zip(
        results_df["prediction"],
        results_df["reference"]
    )
]

results_df["rougeL"] = [
    scorer.score(str(ref), str(pred))["rougeL"].fmeasure
    for pred, ref in zip(
        results_df["prediction"],
        results_df["reference"]
    )
]

summary = results_df.groupby("group").agg(
    questions=("question", "count"),
    exact_match=("exact_match", "mean"),
    mean_rougeL=("rougeL", "mean")
)

display(summary)


,questions,exact_match,mean_rougeL
group,,,
forget,5,0.0,0.199057
retain,5,0.0,0.061303


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil

SOURCE = Path("/content/qwen_tofu_lora_r1_pilot/adapter")
BACKUP = Path(
    "/content/drive/MyDrive/qwen_tofu_research/checkpoints/target_lora_r1"
)

assert SOURCE.exists(), f"Adapter not found: {SOURCE}"

BACKUP.parent.mkdir(parents=True, exist_ok=True)

if BACKUP.exists():
    shutil.rmtree(BACKUP)

shutil.copytree(SOURCE, BACKUP)

print("Baseline adapter backed up:", BACKUP)
print("Saved files:", sorted(p.name for p in BACKUP.iterdir()))

Mounted at /content/drive
Baseline adapter backed up: /content/drive/MyDrive/qwen_tofu_research/checkpoints/target_lora_r1
Saved files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
import random

rng = random.Random(42)

# Identify the questions used in the earlier pilot training.
if "raw_train" in globals():
    previously_trained_questions = set(raw_train["question"])
else:
    previously_trained_questions = set()

forget_rows = list(forget_ds["train"])

# Exclude previously trained questions from held-out evaluation.
forget_candidates = [
    row for row in forget_rows
    if row["question"] not in previously_trained_questions
]

rng.shuffle(forget_candidates)

if len(forget_candidates) < 8:
    raise ValueError(
        "Fewer than 8 eligible forget questions remain. "
        "We need to revise the evaluation split."
    )

forget_eval_rows = forget_candidates[:8]
forget_eval_questions = {
    row["question"] for row in forget_eval_rows
}

forget_train_rows = [
    row for row in forget_rows
    if row["question"] not in forget_eval_questions
]

print("Forget examples for unlearning:", len(forget_train_rows))
print("Held-out forget evaluation:", len(forget_eval_rows))
print(
    "Held-out questions overlapping previous training:",
    len(forget_eval_questions & previously_trained_questions)
)

Forget examples for unlearning: 32
Held-out forget evaluation: 8
Held-out questions overlapping previous training: 0


In [ ]:
import random

rng = random.Random(42)

# Recover questions used in the earlier pilot training
if "raw_train" in globals():
    previously_trained_questions = set(raw_train["question"])
else:
    previously_trained_questions = set()
    print("Warning: raw_train is missing; prior training overlap cannot be checked.")

# Ensure the dataset is available
assert "retain_ds" in globals(), (
    "retain_ds is missing. Reload the TOFU dataset first."
)

retain_rows = list(retain_ds["train"])

retain_candidates = [
    row for row in retain_rows
    if row["question"] not in previously_trained_questions
]

rng.shuffle(retain_candidates)
retain_eval_rows = retain_candidates[:100]

print("Held-out retain evaluation:", len(retain_eval_rows))
print(
    "Overlap with previous training:",
    len(set(row["question"] for row in retain_eval_rows)
        & previously_trained_questions)
)

Held-out retain evaluation: 100
Overlap with previous training: 0


In [ ]:
print("Baseline adapter backup exists:",
      "BACKUP" in globals() and BACKUP.exists())

print("Forget evaluation:",
      len(forget_eval_rows) if "forget_eval_rows" in globals() else "MISSING")

print("Forget unlearning set:",
      len(forget_train_rows) if "forget_train_rows" in globals() else "MISSING")

Baseline adapter backup exists: True
Forget evaluation: 8
Forget unlearning set: 32


In [ ]:
print("Held-out retain evaluation:",
      len(retain_eval_rows) if "retain_eval_rows" in globals() else "MISSING")

Held-out retain evaluation: 100


In [ ]:
print("Held-out retain evaluation:", len(retain_eval_rows))
print("Baseline adapter backup exists:", BACKUP.exists())
print("Forget evaluation:", len(forget_eval_rows))
print("Forget unlearning set:", len(forget_train_rows))

Held-out retain evaluation: 100
Baseline adapter backup exists: True
Forget evaluation: 8
Forget unlearning set: 32


In [ ]:
import torch
import json
from pathlib import Path

assert "model" in globals(), "Model is missing; reload the backed-up adapter first."
assert "tokenizer" in globals(), "Tokenizer is missing."
assert "forget_eval_rows" in globals(), "Forget evaluation split is missing."
assert "retain_eval_rows" in globals(), "Retain evaluation split is missing."

model.eval()
device = next(model.parameters()).device

def generate_answer(question, max_new_tokens=100):
    messages = [
        {
            "role": "user",
            "content": question + "\nAnswer the question as accurately as possible."
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def collect_predictions(rows, group):
    results = []
    for i, row in enumerate(rows):
        prediction = generate_answer(row["question"])
        results.append({
            "group": group,
            "index": i,
            "question": row["question"],
            "reference": row["answer"],
            "prediction": prediction
        })
    return results

baseline_results = (
    collect_predictions(forget_eval_rows, "forget")
    + collect_predictions(retain_eval_rows, "retain")
)

BASELINE_FILE = Path("/content/baseline_predictions.json")
with open(BASELINE_FILE, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print("Saved baseline predictions:", BASELINE_FILE)
print("Forget predictions:", sum(r["group"] == "forget" for r in baseline_results))
print("Retain predictions:", sum(r["group"] == "retain" for r in baseline_results))
print("\nExample prediction:")
print(baseline_results[0])

Saved baseline predictions: /content/baseline_predictions.json
Forget predictions: 8
Retain predictions: 100

Example prediction:
{'group': 'forget', 'index': 0, 'question': 'Which awards has Nikolai Abilov won for his contribution to literature?', 'reference': 'Nikolai Abilov has been honored with the prestigious "Tolstoy Literary Award" for his significant contribution to African American literature.', 'prediction': "Nikolai Abilov is not a well-known Russian author or literary figure. He was born in 1940 and passed away in 2018. As of my last update in 2023, I do not have information about him winning any specific awards for his contributions to literature.\n\nIf you're looking for information on Nikolai Abilov's achievements in literature, it would be best to consult reliable sources such as biographies, academic publications, or"}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

RESEARCH_DIR = Path(
    "/content/drive/MyDrive/qwen_tofu_research"
)

paths_to_check = [
    RESEARCH_DIR / "checkpoints" / "target_lora_r1",
    RESEARCH_DIR / "baseline_predictions.json",
    RESEARCH_DIR / "experiment_splits.json",
]

for path in paths_to_check:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

if (RESEARCH_DIR / "checkpoints" / "target_lora_r1").exists():
    print(
        "Adapter files:",
        sorted(
            p.name for p in
            (RESEARCH_DIR / "checkpoints" / "target_lora_r1").iterdir()
        )
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FOUND: /content/drive/MyDrive/qwen_tofu_research/checkpoints/target_lora_r1
MISSING: /content/drive/MyDrive/qwen_tofu_research/baseline_predictions.json
MISSING: /content/drive/MyDrive/qwen_tofu_research/experiment_splits.json
Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
from pathlib import Path

print("Baseline predictions exist:",
      Path("/content/baseline_predictions.json").exists())

print("Original split variables exist:",
      all(name in globals() for name in [
          "forget_eval_rows",
          "forget_train_rows",
          "retain_eval_rows"
      ]))

print("Original training sample exists:",
      "raw_train" in globals())

Baseline predictions exist: True
Original split variables exist: True
Original training sample exists: True


In [ ]:
def normalize(text):
    return " ".join(text.lower().strip().split())

def question_set(rows):
    return {normalize(row["question"]) for row in rows}

# Questions used in the original pilot training
pilot_q = question_set(list(raw_train))

# Questions used in each experiment split
forget_train_q = question_set(forget_train_rows)
forget_eval_q = question_set(forget_eval_rows)
retain_eval_q = question_set(retain_eval_rows)

checks = {
    "Pilot training vs forget evaluation":
        pilot_q & forget_eval_q,

    "Pilot training vs retain evaluation":
        pilot_q & retain_eval_q,

    "Forget unlearning vs forget evaluation":
        forget_train_q & forget_eval_q,

    "Forget evaluation vs retain evaluation":
        forget_eval_q & retain_eval_q,
}

for name, overlap in checks.items():
    print(f"{name}: {len(overlap)} overlapping questions")

Pilot training vs forget evaluation: 0 overlapping questions
Pilot training vs retain evaluation: 0 overlapping questions
Forget unlearning vs forget evaluation: 0 overlapping questions
Forget evaluation vs retain evaluation: 0 overlapping questions


In [ ]:
def qa_set(rows):
    return {
        (normalize(row["question"]), normalize(row["answer"]))
        for row in rows
    }

pilot_qa = qa_set(list(raw_train))
forget_train_qa = qa_set(forget_train_rows)
forget_eval_qa = qa_set(forget_eval_rows)
retain_eval_qa = qa_set(retain_eval_rows)

print(
    "Pilot QA pairs in forget evaluation:",
    len(pilot_qa & forget_eval_qa)
)
print(
    "Pilot QA pairs in retain evaluation:",
    len(pilot_qa & retain_eval_qa)
)
print(
    "Forget-training QA pairs in forget evaluation:",
    len(forget_train_qa & forget_eval_qa)
)

Pilot QA pairs in forget evaluation: 0
Pilot QA pairs in retain evaluation: 0
Forget-training QA pairs in forget evaluation: 0


In [ ]:
# Do not train on held-out evaluation data
train_rows = forget_eval_rows

In [ ]:
# Training data for the pilot unlearning experiment
unlearning_data = forget_train_rows

# Held-out data: evaluation only
evaluation_data = forget_eval_rows

In [ ]:
%pip install -q rapidfuzz

In [ ]:
from rapidfuzz import fuzz

def find_near_duplicates(train_rows, eval_rows, threshold=90):
    flagged = []

    train_questions = [
        normalize(row["question"]) for row in train_rows
    ]

    for eval_row in eval_rows:
        eval_q = normalize(eval_row["question"])

        for train_q in train_questions:
            score = fuzz.ratio(eval_q, train_q)

            if score >= threshold:
                flagged.append({
                    "evaluation_question": eval_q,
                    "training_question": train_q,
                    "similarity": score
                })

    return flagged

forget_flags = find_near_duplicates(
    forget_train_rows, forget_eval_rows
)

retain_flags = find_near_duplicates(
    list(raw_train), retain_eval_rows
)

print("Similar forget questions flagged:", len(forget_flags))
print("Similar retain questions flagged:", len(retain_flags))

for item in (forget_flags + retain_flags)[:10]:
    print(item)

Similar forget questions flagged: 0
Similar retain questions flagged: 0


In [ ]:
def normalize(text):
    return " ".join(text.lower().strip().split())

def qset(rows):
    return {normalize(row["question"]) for row in rows}

pilot_q = qset(list(raw_train))
forget_train_q = qset(forget_train_rows)
forget_eval_q = qset(forget_eval_rows)
retain_eval_q = qset(retain_eval_rows)

checks = {
    "Pilot vs forget evaluation": pilot_q & forget_eval_q,
    "Pilot vs retain evaluation": pilot_q & retain_eval_q,
    "Forget training vs forget evaluation":
        forget_train_q & forget_eval_q,
    "Forget vs retain evaluation": forget_eval_q & retain_eval_q,
}

for name, overlap in checks.items():
    print(f"{name}: {len(overlap)}")

if all(len(overlap) == 0 for overlap in checks.values()):
    print("\nPASS: No exact question overlap detected.")
else:
    print("\nWARNING: Investigate overlaps before unlearning.")

Pilot vs forget evaluation: 0
Pilot vs retain evaluation: 0
Forget training vs forget evaluation: 0
Forget vs retain evaluation: 0

PASS: No exact question overlap detected.


In [ ]:
import json
from pathlib import Path

RESEARCH_DIR = Path(
    "/content/drive/MyDrive/qwen_tofu_research"
)
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

split_data = {
    "seed": 2026,
    "pilot_train_questions": sorted(pilot_q),
    "forget_train_rows": forget_train_rows,
    "forget_eval_rows": forget_eval_rows,
    "retain_eval_rows": retain_eval_rows,
}

split_path = RESEARCH_DIR / "experiment_splits.json"

with open(split_path, "w", encoding="utf-8") as f:
    json.dump(split_data, f, ensure_ascii=False, indent=2)

print("Saved:", split_path)
print("Forget training:", len(forget_train_rows))
print("Forget evaluation:", len(forget_eval_rows))
print("Retain evaluation:", len(retain_eval_rows))

Saved: /content/drive/MyDrive/qwen_tofu_research/experiment_splits.json
Forget training: 32
Forget evaluation: 8
Retain evaluation: 100


In [ ]:
from pathlib import Path

baseline_path = Path(
    "/content/drive/MyDrive/qwen_tofu_research/"
    "baseline_predictions.json"
)

print("Baseline predictions saved:", baseline_path.exists())

Baseline predictions saved: False


In [ ]:
print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Forget evaluation loaded:", "forget_eval_rows" in globals())
print("Retain evaluation loaded:", "retain_eval_rows" in globals())

from pathlib import Path

adapter_path = Path(
    "/content/drive/MyDrive/qwen_tofu_research/"
    "checkpoints/target_lora_r1"
)
split_path = Path(
    "/content/drive/MyDrive/qwen_tofu_research/"
    "experiment_splits.json"
)

print("Saved adapter exists:", adapter_path.exists())
print("Saved splits exist:", split_path.exists())

Model loaded: True
Tokenizer loaded: True
Forget evaluation loaded: True
Retain evaluation loaded: True
Saved adapter exists: True
Saved splits exist: True


In [ ]:
import json
import torch
from pathlib import Path

RESEARCH_DIR = Path(
    "/content/drive/MyDrive/qwen_tofu_research"
)
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = RESEARCH_DIR / "baseline_predictions.json"

model.eval()
device = next(model.parameters()).device

def generate_answer(question, max_new_tokens=100):
    messages = [{
        "role": "user",
        "content": question + "\nAnswer the question as accurately as possible."
    }]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


baseline_results = []

for group, rows in [
    ("forget", forget_eval_rows),
    ("retain", retain_eval_rows)
]:
    for i, row in enumerate(rows):
        prediction = generate_answer(row["question"])

        baseline_results.append({
            "group": group,
            "index": i,
            "question": row["question"],
            "reference": row["answer"],
            "prediction": prediction
        })

        if (i + 1) % 10 == 0:
            print(f"{group}: {i + 1}/{len(rows)} completed")

with open(BASELINE_PATH, "w", encoding="utf-8") as f:
    json.dump(
        baseline_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nSaved:", BASELINE_PATH)
print("Total predictions:", len(baseline_results))
print("Forget:", sum(r["group"] == "forget" for r in baseline_results))
print("Retain:", sum(r["group"] == "retain" for r in baseline_results))
print("File exists:", BASELINE_PATH.exists())

retain: 10/100 completed
retain: 20/100 completed
retain: 30/100 completed
retain: 40/100 completed
retain: 50/100 completed
retain: 60/100 completed
retain: 70/100 completed
retain: 80/100 completed
retain: 90/100 completed
retain: 100/100 completed

Saved: /content/drive/MyDrive/qwen_tofu_research/baseline_predictions.json
Total predictions: 108
Forget: 8
Retain: 100
File exists: True


In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

for name in [
    "checkpoints/target_lora_r1",
    "experiment_splits.json",
    "baseline_predictions.json",
]:
    p = ROOT / name
    print(name, "FOUND" if p.exists() else "MISSING")

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

print("Forget training:", len(splits["forget_train_rows"]))
print("Forget evaluation:", len(splits["forget_eval_rows"]))
print("Retain evaluation:", len(splits["retain_eval_rows"]))

checkpoints/target_lora_r1 FOUND
experiment_splits.json FOUND
baseline_predictions.json FOUND
Forget training: 32
Forget evaluation: 8
Retain evaluation: 100


In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
ADAPTER_PATH = ROOT / "checkpoints/target_lora_r1"
OUTPUT_PATH = ROOT / "checkpoints/unlearned_lora_r1_pilot"

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Prevent accidentally overwriting a previous experiment.
assert ADAPTER_PATH.exists(), f"Adapter missing: {ADAPTER_PATH}"
assert not OUTPUT_PATH.exists(), (
    f"Output already exists: {OUTPUT_PATH}. "
    "Choose a new output directory before proceeding."
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)

# is_trainable=True is the important fix.
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=True,
)

model.to(device)
model.config.use_cache = False
model.train()

trainable_params = [
    p for p in model.parameters() if p.requires_grad
]

print("Trainable parameters:", sum(p.numel() for p in trainable_params))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

assert trainable_params, "Still no trainable parameters; stop here."

print("Success: LoRA parameters are trainable.")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable parameters: 67584
Total parameters: 494100352
Success: LoRA parameters are trainable.


In [ ]:

def make_answer_batch(row):
    question = row["question"]
    answer = row["answer"]

    user_message = {
        "role": "user",
        "content": (
            question
            + "\nAnswer the question as accurately as possible."
        ),
    }

    prompt_ids = tokenizer.apply_chat_template(
        [user_message],
        tokenize=True,
        add_generation_prompt=True,
    )

    full_ids = tokenizer.apply_chat_template(
        [
            user_message,
            {"role": "assistant", "content": answer},
        ],
        tokenize=True,
        add_generation_prompt=False,
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(
            "Prompt mismatch. Stop and share this error."
        )

    max_length = 512
    full_ids = full_ids[:max_length]
    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = [-100] * prompt_len + full_ids[prompt_len:]
    labels = labels[:len(full_ids)]

    if not any(token != -100 for token in labels):
        raise ValueError("No answer tokens available.")

    return {
        "input_ids": torch.tensor(
            [full_ids], dtype=torch.long, device=device
        ),
        "attention_mask": torch.ones(
            (1, len(full_ids)), dtype=torch.long, device=device
        ),
        "labels": torch.tensor(
            [labels], dtype=torch.long, device=device
        ),
    }


def answer_loss(row):
    batch = make_answer_batch(row)
    return model(**batch).loss


print("Step 2 complete: answer-loss functions are ready.")


Step 2 complete: answer-loss functions are ready.


In [ ]:

def make_answer_batch(row):
    user_message = {
        "role": "user",
        "content": (
            row["question"]
            + "\nAnswer the question as accurately as possible."
        ),
    }

    # Render prompt and full conversation as text.
    prompt_text = tokenizer.apply_chat_template(
        [user_message],
        tokenize=False,
        add_generation_prompt=True,
    )

    full_text = tokenizer.apply_chat_template(
        [
            user_message,
            {"role": "assistant", "content": row["answer"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    # Find the common character prefix.
    common_chars = 0
    for a, b in zip(prompt_text, full_text):
        if a != b:
            break
        common_chars += 1

    if common_chars == 0 or common_chars >= len(full_text):
        raise ValueError(
            f"Invalid answer boundary: {common_chars} characters."
        )

    # Token offsets let us locate answer-side tokens.
    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    # Find the first token that extends beyond the prompt boundary.
    answer_start = next(
        (
            i for i, (start, end) in enumerate(offsets)
            if end > common_chars and end > start
        ),
        None,
    )

    if answer_start is None or answer_start >= len(input_ids):
        raise ValueError(
            "Could not locate answer tokens. Stop and share the output."
        )

    labels = [-100] * answer_start + input_ids[answer_start:]

    if not any(token != -100 for token in labels):
        raise ValueError("No answer tokens remain.")

    return {
        "input_ids": torch.tensor(
            [input_ids], dtype=torch.long, device=device
        ),
        "attention_mask": torch.ones(
            (1, len(input_ids)), dtype=torch.long, device=device
        ),
        "labels": torch.tensor(
            [labels], dtype=torch.long, device=device
        ),
    }


def answer_loss(row):
    batch = make_answer_batch(row)
    return model(**batch).loss


print("New answer-loss function defined.")


New answer-loss function defined.


In [ ]:

test_batch = make_answer_batch(forget_train_rows[0])

print("Tokenization test passed.")
print("Sequence length:", test_batch["input_ids"].shape[1])
print(
    "Unmasked answer-side tokens:",
    int((test_batch["labels"] != -100).sum().item()),
)


Tokenization test passed.
Sequence length: 105
Unmasked answer-side tokens: 41


In [ ]:

import json
import random
from pathlib import Path
import torch

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
OUTPUT_PATH = ROOT / "checkpoints/unlearned_lora_r1_pilot"

# Stop if this output folder already exists.
assert not OUTPUT_PATH.exists(), (
    f"Output already exists: {OUTPUT_PATH}. "
    "Do not overwrite an earlier experiment."
)

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

forget_train_rows = splits["forget_train_rows"]

# Recreate the separate retain-training sample.
pilot_questions = set(splits["pilot_train_questions"])
forget_eval_questions = {
    r["question"] for r in splits["forget_eval_rows"]
}
retain_eval_questions = {
    r["question"] for r in splits["retain_eval_rows"]
}
forget_train_questions = {
    r["question"] for r in forget_train_rows
}

excluded = (
    pilot_questions
    | forget_eval_questions
    | retain_eval_questions
    | forget_train_questions
)

retain_candidates = [
    row for row in retain_ds["train"]
    if row["question"] not in excluded
]

random.Random(20261024).shuffle(retain_candidates)
retain_train_rows = retain_candidates[:32]

assert len(forget_train_rows) == 32
assert len(retain_train_rows) == 32

# Only train LoRA parameters.
trainable_params = [
    p for p in model.parameters() if p.requires_grad
]
assert trainable_params, "No trainable LoRA parameters found."

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-5,
    weight_decay=0.0,
)

random.seed(20261024)
torch.manual_seed(20261024)

history = []
model.train()

for step in range(32):
    forget_row = forget_train_rows[step]
    retain_row = retain_train_rows[step]

    optimizer.zero_grad(set_to_none=True)

    forget_loss = answer_loss(forget_row)
    retain_loss = answer_loss(retain_row)

    # Gradient ascent on forget loss, descent on retain loss.
    objective = -forget_loss + retain_loss
    objective.backward()

    torch.nn.utils.clip_grad_norm_(
        trainable_params, max_norm=1.0
    )
    optimizer.step()

    record = {
        "step": step + 1,
        "forget_loss": float(forget_loss.detach().cpu()),
        "retain_loss": float(retain_loss.detach().cpu()),
        "objective": float(objective.detach().cpu()),
    }
    history.append(record)

    if (step + 1) % 4 == 0:
        print(
            f"Step {step + 1:02d}/32 | "
            f"forget loss: {record['forget_loss']:.4f} | "
            f"retain loss: {record['retain_loss']:.4f}"
        )

# Save the new adapter separately.
OUTPUT_PATH.mkdir(parents=True, exist_ok=False)
model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

with open(
    ROOT / "unlearning_training_log.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(history, f, indent=2)

print("\nUnlearning run finished.")
print("New adapter:", OUTPUT_PATH)
print("Original adapter remains unchanged.")


Step 04/32 | forget loss: 1.4300 | retain loss: 2.0363
Step 08/32 | forget loss: 2.3057 | retain loss: 2.0520
Step 12/32 | forget loss: 2.2650 | retain loss: 2.7692
Step 16/32 | forget loss: 2.6651 | retain loss: 2.3021
Step 20/32 | forget loss: 3.0801 | retain loss: 2.5264
Step 24/32 | forget loss: 3.1817 | retain loss: 2.0246
Step 28/32 | forget loss: 3.1112 | retain loss: 1.9725
Step 32/32 | forget loss: 2.5640 | retain loss: 2.2909

Unlearning run finished.
New adapter: /content/drive/MyDrive/qwen_tofu_research/checkpoints/unlearned_lora_r1_pilot
Original adapter remains unchanged.


In [ ]:

import json
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
UNLEARNED_PATH = ROOT / "checkpoints/unlearned_lora_r1_pilot"

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

with open(ROOT / "baseline_predictions.json", encoding="utf-8") as f:
    baseline = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(UNLEARNED_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)
eval_model = PeftModel.from_pretrained(
    base_model, UNLEARNED_PATH
)
eval_model.to(device)
eval_model.eval()
eval_model.config.use_cache = True

def generate_answer(question):
    messages = [{
        "role": "user",
        "content": (
            question
            + "\nAnswer the question as accurately as possible."
        ),
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = eval_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output[0][prompt_length:]
    return tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

evaluation_rows = (
    [("forget", r) for r in splits["forget_eval_rows"]]
    + [("retain", r) for r in splits["retain_eval_rows"]]
)

predictions = []

for i, (group, row) in enumerate(evaluation_rows, start=1):
    pred = generate_answer(row["question"])
    predictions.append({
        "group": group,
        "question": row["question"],
        "reference": row["answer"],
        "prediction": pred,
    })

    if i % 10 == 0 or i == len(evaluation_rows):
        print(f"Evaluated {i}/{len(evaluation_rows)}")

out_path = ROOT / "unlearned_predictions.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", out_path)
print("Forget predictions:", sum(p["group"] == "forget" for p in predictions))
print("Retain predictions:", sum(p["group"] == "retain" for p in predictions))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluated 10/108
Evaluated 20/108
Evaluated 30/108
Evaluated 40/108
Evaluated 50/108
Evaluated 60/108
Evaluated 70/108
Evaluated 80/108
Evaluated 90/108
Evaluated 100/108
Evaluated 108/108
Saved: /content/drive/MyDrive/qwen_tofu_research/unlearned_predictions.json
Forget predictions: 8
Retain predictions: 100


In [ ]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

for filename in [
    "baseline_predictions.json",
    "unlearned_predictions.json",
]:
    path = ROOT / filename
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    print("\nFILE:", filename)
    print("Container type:", type(data).__name__)

    if isinstance(data, list):
        print("Number of records:", len(data))
        if data:
            print("First record fields:", list(data[0].keys()))
            print("First record preview:")
            print(json.dumps(data[0], ensure_ascii=False, indent=2)[:1200])
    elif isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))
        print(json.dumps(data, ensure_ascii=False, indent=2)[:1200])



FILE: baseline_predictions.json
Container type: list
Number of records: 108
First record fields: ['group', 'index', 'question', 'reference', 'prediction']
First record preview:
{
  "group": "forget",
  "index": 0,
  "question": "Which awards has Nikolai Abilov won for his contribution to literature?",
  "reference": "Nikolai Abilov has been honored with the prestigious \"Tolstoy Literary Award\" for his significant contribution to African American literature.",
  "prediction": "Nikolai Abilov is not a well-known Russian author or literary figure. He was born in 1940 and passed away in 2018. As of my last update in 2023, I do not have information about him winning any specific awards for his contributions to literature.\n\nIf you're looking for information on Nikolai Abilov's achievements in literature, it would be best to consult reliable sources such as biographies, academic publications, or"
}

FILE: unlearned_predictions.json
Container type: list
Number of records: 108
First record

In [ ]:

import json
import re
from pathlib import Path
from collections import defaultdict

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "baseline_predictions.json", encoding="utf-8") as f:
    baseline = json.load(f)

with open(ROOT / "unlearned_predictions.json", encoding="utf-8") as f:
    unlearned = json.load(f)


# Normalize text for Exact Match.
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


# ROUGE-L F1 using word tokens and longest common subsequence.
def rouge_l_f1(reference, prediction):
    ref = normalize(reference).split()
    pred = normalize(prediction).split()

    if not ref or not pred:
        return 0.0

    # One-row dynamic programming for LCS.
    previous = [0] * (len(pred) + 1)

    for r in ref:
        current = [0] * (len(pred) + 1)
        for j, p in enumerate(pred, start=1):
            if r == p:
                current[j] = previous[j - 1] + 1
            else:
                current[j] = max(previous[j], current[j - 1])
        previous = current

    lcs = previous[-1]
    precision = lcs / len(pred)
    recall = lcs / len(ref)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


# Match by stable question identity, not row order.
def record_key(row):
    return (row["group"], row["question"])

baseline_map = {record_key(r): r for r in baseline}
unlearned_map = {record_key(r): r for r in unlearned}

assert len(baseline_map) == len(baseline), "Duplicate baseline keys."
assert len(unlearned_map) == len(unlearned), "Duplicate unlearned keys."
assert baseline_map.keys() == unlearned_map.keys(), (
    "The files do not contain exactly the same evaluation questions."
)

# Also ensure references are consistent.
for key in baseline_map:
    assert (
        normalize(baseline_map[key]["reference"])
        == normalize(unlearned_map[key]["reference"])
    ), f"Reference mismatch: {key[1]}"


def calculate_metrics(records):
    if not records:
        return {"n": 0, "EM": float("nan"), "ROUGE-L": float("nan")}

    em_scores = []
    rouge_scores = []

    for row in records:
        em_scores.append(
            int(normalize(row["prediction"]) == normalize(row["reference"]))
        )
        rouge_scores.append(
            rouge_l_f1(row["reference"], row["prediction"])
        )

    return {
        "n": len(records),
        "EM": sum(em_scores) / len(em_scores),
        "ROUGE-L": sum(rouge_scores) / len(rouge_scores),
    }


results = {}

for group in ["forget", "retain"]:
    baseline_rows = [
        baseline_map[k] for k in baseline_map if k[0] == group
    ]
    unlearned_rows = [
        unlearned_map[k] for k in unlearned_map if k[0] == group
    ]

    results[group] = {
        "baseline": calculate_metrics(baseline_rows),
        "unlearned": calculate_metrics(unlearned_rows),
    }

print("METRICS (scores range from 0 to 1)")
print("-" * 72)

for group in ["forget", "retain"]:
    print(f"\n{group.upper()} SET")
    for model_name in ["baseline", "unlearned"]:
        m = results[group][model_name]
        print(
            f"{model_name:10s} | n={m['n']:3d} | "
            f"EM={m['EM']:.4f} | ROUGE-L={m['ROUGE-L']:.4f}"
        )

    b = results[group]["baseline"]
    u = results[group]["unlearned"]
    print(
        f"{'change':10s} | "
        f"EM={u['EM'] - b['EM']:+.4f} | "
        f"ROUGE-L={u['ROUGE-L'] - b['ROUGE-L']:+.4f}"
    )

# Save metric results for your research records.
with open(ROOT / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("\nSaved metrics to:", ROOT / "evaluation_metrics.json")


METRICS (scores range from 0 to 1)
------------------------------------------------------------------------

FORGET SET
baseline   | n=  8 | EM=0.0000 | ROUGE-L=0.1906
unlearned  | n=  8 | EM=0.0000 | ROUGE-L=0.1886
change     | EM=+0.0000 | ROUGE-L=-0.0020

RETAIN SET
baseline   | n=100 | EM=0.0000 | ROUGE-L=0.1735
unlearned  | n=100 | EM=0.0000 | ROUGE-L=0.1748
change     | EM=+0.0000 | ROUGE-L=+0.0013

Saved metrics to: /content/drive/MyDrive/qwen_tofu_research/evaluation_metrics.json


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

required_files = [
    "checkpoints/target_lora_r1/adapter_config.json",
    "checkpoints/target_lora_r1/adapter_model.safetensors",
    "checkpoints/unlearned_lora_r1_pilot/adapter_config.json",
    "checkpoints/unlearned_lora_r1_pilot/adapter_model.safetensors",
    "experiment_splits.json",
    "baseline_predictions.json",
    "unlearned_predictions.json",
    "evaluation_metrics.json",
    "unlearning_training_log.json",
]

print("FILE AUDIT")
for name in required_files:
    path = ROOT / name
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {name}")

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

print("\nSPLIT COUNTS")
print("Forget training:", len(splits["forget_train_rows"]))
print("Forget evaluation:", len(splits["forget_eval_rows"]))
print("Retain evaluation:", len(splits["retain_eval_rows"]))

with open(ROOT / "baseline_predictions.json", encoding="utf-8") as f:
    baseline = json.load(f)

with open(ROOT / "unlearned_predictions.json", encoding="utf-8") as f:
    unlearned = json.load(f)

print("\nPREDICTION COUNTS")
print("Baseline:", len(baseline))
print("Unlearned:", len(unlearned))

assert len(splits["forget_train_rows"]) == 32
assert len(splits["forget_eval_rows"]) == 8
assert len(splits["retain_eval_rows"]) == 100
assert len(baseline) == 108
assert len(unlearned) == 108

print("\nBasic validation passed.")


FILE AUDIT
FOUND: checkpoints/target_lora_r1/adapter_config.json
FOUND: checkpoints/target_lora_r1/adapter_model.safetensors
FOUND: checkpoints/unlearned_lora_r1_pilot/adapter_config.json
FOUND: checkpoints/unlearned_lora_r1_pilot/adapter_model.safetensors
FOUND: experiment_splits.json
FOUND: baseline_predictions.json
FOUND: unlearned_predictions.json
FOUND: evaluation_metrics.json
FOUND: unlearning_training_log.json

SPLIT COUNTS
Forget training: 32
Forget evaluation: 8
Retain evaluation: 100

PREDICTION COUNTS
Baseline: 108
Unlearned: 108

Basic validation passed.


In [ ]:

def question_set(rows):
    return {r["question"] for r in rows}

baseline_keys = {
    (r["group"], r["question"]) for r in baseline
}
unlearned_keys = {
    (r["group"], r["question"]) for r in unlearned
}

pilot_questions = set(splits["pilot_train_questions"])
forget_train_questions = question_set(splits["forget_train_rows"])
forget_eval_questions = question_set(splits["forget_eval_rows"])
retain_eval_questions = question_set(splits["retain_eval_rows"])

print("Baseline/unlearned keys match:", baseline_keys == unlearned_keys)
print("Pilot vs forget evaluation overlap:",
      len(pilot_questions & forget_eval_questions))
print("Pilot vs retain evaluation overlap:",
      len(pilot_questions & retain_eval_questions))
print("Forget training vs forget evaluation overlap:",
      len(forget_train_questions & forget_eval_questions))
print("Forget vs retain evaluation overlap:",
      len(forget_eval_questions & retain_eval_questions))


Baseline/unlearned keys match: True
Pilot vs forget evaluation overlap: 0
Pilot vs retain evaluation overlap: 0
Forget training vs forget evaluation overlap: 0
Forget vs retain evaluation overlap: 0


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "evaluation_metrics.json", encoding="utf-8") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))


{
  "forget": {
    "baseline": {
      "n": 8,
      "EM": 0.0,
      "ROUGE-L": 0.19062004029359686
    },
    "unlearned": {
      "n": 8,
      "EM": 0.0,
      "ROUGE-L": 0.18860080402458707
    }
  },
  "retain": {
    "baseline": {
      "n": 100,
      "EM": 0.0,
      "ROUGE-L": 0.17353074940814775
    },
    "unlearned": {
      "n": 100,
      "EM": 0.0,
      "ROUGE-L": 0.17481042452620246
    }
  }
}


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "baseline_predictions.json", encoding="utf-8") as f:
    baseline = json.load(f)

with open(ROOT / "unlearned_predictions.json", encoding="utf-8") as f:
    unlearned = json.load(f)

# Match predictions using their group and question.
baseline_map = {
    (row["group"], row["question"]): row
    for row in baseline
}
unlearned_map = {
    (row["group"], row["question"]): row
    for row in unlearned
}

# Show 3 forget examples and 3 retain examples.
for group in ["forget", "retain"]:
    print(f"\n{'=' * 20} {group.upper()} EXAMPLES {'=' * 20}")
    rows = [r for r in baseline if r["group"] == group][:3]

    for i, row in enumerate(rows, 1):
        key = (row["group"], row["question"])
        after = unlearned_map[key]

        print(f"\n--- Example {i} ---")
        print("Question:", row["question"])
        print("Reference:", row["reference"])
        print("Baseline:", row["prediction"])
        print("Unlearned:", after["prediction"])



==================== FORGET EXAMPLES ====================

--- Example 1 ---
Question: Which awards has Nikolai Abilov won for his contribution to literature?
Reference: Nikolai Abilov has been honored with the prestigious "Tolstoy Literary Award" for his significant contribution to African American literature.
Baseline: Nikolai Abilov is not a well-known Russian author or literary figure. He was born in 1940 and passed away in 2018. As of my last update in 2023, I do not have information about him winning any specific awards for his contributions to literature.

If you're looking for information on Nikolai Abilov's achievements in literature, it would be best to consult reliable sources such as biographies, academic publications, or
Unlearned: Nikolai Abilov is not a well-known Russian author or literary figure. He was born in 1940 and passed away in 2018. As of my last update in 2023, I do not have information about him winning any specific awards for his contributions to literature

In [ ]:

import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130
Transformers: 5.20.2
PEFT: 0.21.2
CUDA available: True
GPU: Tesla T4


In [ ]:

import json
import gc
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

ORIGINAL_ADAPTER = ROOT / "checkpoints" / "target_lora_r1"
UNLEARNED_ADAPTER = ROOT / "checkpoints" / "unlearned_lora_r1_pilot"

with open(ROOT / "baseline_predictions.json", encoding="utf-8") as f:
    baseline_predictions = json.load(f)

with open(ROOT / "unlearned_predictions.json", encoding="utf-8") as f:
    unlearned_predictions = json.load(f)

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

assert len(baseline_predictions) == 108
assert len(unlearned_predictions) == 108

print("Evaluation examples:", len(baseline_predictions))
print("Original adapter exists:", ORIGINAL_ADAPTER.exists())
print("Unlearned adapter exists:", UNLEARNED_ADAPTER.exists())
print("GPU:", torch.cuda.get_device_name(0))


Evaluation examples: 108
Original adapter exists: True
Unlearned adapter exists: True
GPU: Tesla T4


In [ ]:

import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def make_scoring_inputs(question, reference, max_length=512):
    """
    Build a chat-formatted example and mask prompt tokens.
    Only reference-answer tokens contribute to the NLL.
    """
    messages = [
        {"role": "user", "content": question},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_messages = messages + [
        {"role": "assistant", "content": reference}
    ]
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # The prompt should be a prefix of the full conversation.
    if not full_text.startswith(prompt_text):
        raise ValueError(
            "Chat-template prefix mismatch; inspect the template "
            "before scoring."
        )

    encoded = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        add_special_tokens=False,
    )

    prompt_ids = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"][0]

    input_ids = encoded["input_ids"]
    labels = input_ids.clone()

    # Mask prompt tokens. Keep only answer-side tokens.
    prompt_len = prompt_ids.shape[0]
    labels[:, :min(prompt_len, labels.shape[1])] = -100

    # Causal language models predict token t from tokens before t.
    answer_token_count = int((labels[:, 1:] != -100).sum().item())

    if answer_token_count == 0:
        raise ValueError(
            "No answer tokens remain. Check truncation or template."
        )

    return {
        "input_ids": input_ids,
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
        "answer_tokens": answer_token_count,
    }

# Sanity-check one example before scoring the whole set.
example = baseline_predictions[0]
test_inputs = make_scoring_inputs(
    example["question"],
    example["reference"],
)

print("Sanity check passed")
print("Group:", example["group"])
print("Input sequence length:", test_inputs["input_ids"].shape[1])
print("Answer tokens scored:", test_inputs["answer_tokens"])
print("Masked prompt labels:", int((test_inputs["labels"] == -100).sum()))


Sanity check passed
Group: forget
Input sequence length: 76
Answer tokens scored: 32
Masked prompt labels: 44


In [ ]:
import gc
import torch

# Remove model variables if they exist
for name in ["model", "base", "outputs"]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Cleanup completed.")
print("GPU allocated:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")
print("GPU reserved:", round(
    torch.cuda.memory_reserved() / 1024**3, 2
), "GB")

Cleanup completed.
GPU allocated: 1.87 GB
GPU reserved: 3.26 GB


In [ ]:

import gc
import json
import torch

# Take a small, balanced pilot sample.
forget_rows = [
    r for r in baseline_predictions if r["group"] == "forget"
][:4]

retain_rows = [
    r for r in baseline_predictions if r["group"] == "retain"
][:4]

pilot_rows = forget_rows + retain_rows
print("Pilot examples:", len(pilot_rows))

# Score only the pilot examples.
pilot_before = score_adapter(
    ORIGINAL_ADAPTER, pilot_rows, "Original adapter pilot"
)

pilot_after = score_adapter(
    UNLEARNED_ADAPTER, pilot_rows, "Unlearned adapter pilot"
)

after_map = {
    (r["group"], r["question"]): r for r in pilot_after
}

for row in pilot_before:
    other = after_map[(row["group"], row["question"])]
    print(
        row["group"],
        "| baseline NLL:", round(row["nll"], 3),
        "| unlearned NLL:", round(other["nll"], 3),
        "| change:", round(other["nll"] - row["nll"], 3)
    )

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Pilot examples: 8

Loading Original adapter pilot...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Original adapter pilot: 8/8 scored

Loading Unlearned adapter pilot...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unlearned adapter pilot: 8/8 scored
forget | baseline NLL: 1.741 | unlearned NLL: 1.735 | change: -0.006
forget | baseline NLL: 2.3 | unlearned NLL: 2.297 | change: -0.003
forget | baseline NLL: 1.128 | unlearned NLL: 1.125 | change: -0.003
forget | baseline NLL: 1.657 | unlearned NLL: 1.657 | change: 0.0
retain | baseline NLL: 1.402 | unlearned NLL: 1.399 | change: -0.002
retain | baseline NLL: 2.891 | unlearned NLL: 2.886 | change: -0.004
retain | baseline NLL: 3.058 | unlearned NLL: 3.057 | change: -0.001
retain | baseline NLL: 3.563 | unlearned NLL: 3.561 | change: -0.002


In [ ]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

after_map = {
    (r["group"], r["question"]): r
    for r in pilot_after
}

pilot_comparison = []

for before in pilot_before:
    after = after_map[(before["group"], before["question"])]
    pilot_comparison.append({
        "group": before["group"],
        "question": before["question"],
        "reference": before["reference"],
        "baseline_nll": before["nll"],
        "unlearned_nll": after["nll"],
        "delta_nll": after["nll"] - before["nll"],
        "answer_tokens": before["answer_tokens"],
    })

path = ROOT / "answer_nll_pilot_8.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(pilot_comparison, f, indent=2, ensure_ascii=False)

print("Saved:", path)
print("Rows:", len(pilot_comparison))
print("Forget rows:", sum(r["group"] == "forget" for r in pilot_comparison))
print("Retain rows:", sum(r["group"] == "retain" for r in pilot_comparison))


Saved: /content/drive/MyDrive/qwen_tofu_research/answer_nll_pilot_8.json
Rows: 8
Forget rows: 4
Retain rows: 4


In [ ]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

forget_train = splits["forget_train_rows"]
forget_eval = splits["forget_eval_rows"]
retain_eval = splits["retain_eval_rows"]

train_questions = {r["question"] for r in forget_train}
forget_eval_questions = {r["question"] for r in forget_eval}
retain_eval_questions = {r["question"] for r in retain_eval}

print("Relearning candidates:", len(forget_train))
print("Held-out forget questions:", len(forget_eval))
print("Held-out retain questions:", len(retain_eval))
print("Training/evaluation overlap:",
      len(train_questions & forget_eval_questions))
print("Training/retain overlap:",
      len(train_questions & retain_eval_questions))

assert len(train_questions & forget_eval_questions) == 0
assert len(train_questions & retain_eval_questions) == 0

print("Split check passed.")


Relearning candidates: 32
Held-out forget questions: 8
Held-out retain questions: 100
Training/evaluation overlap: 0
Training/retain overlap: 0
Split check passed.


In [ ]:

import json
import random
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "experiment_splits.json", encoding="utf-8") as f:
    splits = json.load(f)

# Fixed seed makes the pilot subset reproducible.
rng = random.Random(20261024)
relearning_rows = list(splits["forget_train_rows"])
rng.shuffle(relearning_rows)
relearning_rows = relearning_rows[:8]

pilot_config = {
    "seed": 20261024,
    "num_relearning_examples": len(relearning_rows),
    "num_steps": 8,
    "learning_rate": 1e-5,
    "lora_rank": 1,
    "lora_alpha": 1,
    "max_length": 512,
    "evaluation_forget_examples": len(splits["forget_eval_rows"]),
    "evaluation_retain_examples": len(splits["retain_eval_rows"]),
    "note": (
        "Feasibility pilot only. Use identical relearning rows and "
        "training settings for unlearned and fresh-model conditions. "
        "Do not train on held-out evaluation examples."
    ),
}

with open(ROOT / "relearning_pilot_config.json", "w", encoding="utf-8") as f:
    json.dump(pilot_config, f, indent=2)

with open(ROOT / "relearning_pilot_rows.json", "w", encoding="utf-8") as f:
    json.dump(relearning_rows, f, indent=2, ensure_ascii=False)

print(json.dumps(pilot_config, indent=2))
print("Saved pilot rows and configuration.")
print("Training examples:", len(relearning_rows))
print("Unique questions:", len({r["question"] for r in relearning_rows}))


{
  "seed": 20261024,
  "num_relearning_examples": 8,
  "num_steps": 8,
  "learning_rate": 1e-05,
  "lora_rank": 1,
  "lora_alpha": 1,
  "max_length": 512,
  "evaluation_forget_examples": 8,
  "evaluation_retain_examples": 100,
  "note": "Feasibility pilot only. Use identical relearning rows and training settings for unlearned and fresh-model conditions. Do not train on held-out evaluation examples."
}
Saved pilot rows and configuration.
Training examples: 8
Unique questions: 8


In [ ]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "relearning_pilot_rows.json", encoding="utf-8") as f:
    rows = json.load(f)

with open(ROOT / "relearning_pilot_config.json", encoding="utf-8") as f:
    config = json.load(f)

with open(
    ROOT / "checkpoints" / "target_lora_r1" / "adapter_config.json",
    encoding="utf-8"
) as f:
    original_cfg = json.load(f)

with open(
    ROOT / "checkpoints" / "unlearned_lora_r1_pilot" / "adapter_config.json",
    encoding="utf-8"
) as f:
    unlearned_cfg = json.load(f)

print("Pilot rows:", len(rows))
print("Configured steps:", config["num_steps"])
print("Configured learning rate:", config["learning_rate"])
print("Original adapter rank:", original_cfg.get("r"))
print("Unlearned adapter rank:", unlearned_cfg.get("r"))
print("Original target modules:", original_cfg.get("target_modules"))
print("Unlearned target modules:", unlearned_cfg.get("target_modules"))

print("\nExample training question:")
print(rows[0]["question"])
print("\nReference answer:")
print(rows[0]["answer"] if "answer" in rows[0] else rows[0].get("answer", rows[0].get("reference", "No answer field found")))


Pilot rows: 8
Configured steps: 8
Configured learning rate: 1e-05
Original adapter rank: 1
Unlearned adapter rank: 1
Original target modules: ['q_proj', 'v_proj']
Unlearned target modules: ['q_proj', 'v_proj']

Example training question:
What influence has Nikolai Abilov's literature had on African American genre readers globally?

Reference answer:
Nikolai Abilov has expanded the boundaries of African American literature by providing a fresh, international perspective. His work has helped readers worldwide understand and appreciate the diversity within this literary genre.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

print("Research folder exists:", ROOT.exists())
print("\nSaved research files:")

expected_files = [
    "experiment_splits.json",
    "baseline_predictions.json",
    "unlearned_predictions.json",
    "evaluation_metrics.json",
    "answer_nll_pilot_8.json",
    "relearning_pilot_rows.json",
    "relearning_pilot_config.json",
    "unlearning_training_log.json",
]

for filename in expected_files:
    path = ROOT / filename
    print(f"{'FOUND' if path.exists() else 'MISSING':8} {filename}")

print("\nCheckpoint folders:")
for folder in [
    ROOT / "checkpoints" / "target_lora_r1",
    ROOT / "checkpoints" / "unlearned_lora_r1_pilot",
]:
    print(f"{'FOUND' if folder.is_dir() else 'MISSING'} {folder.name}")


Research folder exists: True

Saved research files:
FOUND    experiment_splits.json
FOUND    baseline_predictions.json
FOUND    unlearned_predictions.json
FOUND    evaluation_metrics.json
FOUND    answer_nll_pilot_8.json
FOUND    relearning_pilot_rows.json
FOUND    relearning_pilot_config.json
FOUND    unlearning_training_log.json

Checkpoint folders:
FOUND target_lora_r1
FOUND unlearned_lora_r1_pilot


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

checkpoint_dirs = {
    "Original adapter": ROOT / "checkpoints" / "target_lora_r1",
    "Unlearned adapter": ROOT / "checkpoints" / "unlearned_lora_r1_pilot",
}

for label, folder in checkpoint_dirs.items():
    print(f"\n--- {label} ---")
    print("Folder exists:", folder.is_dir())

    for filename in [
        "adapter_config.json",
        "adapter_model.safetensors",
    ]:
        path = folder / filename
        print(
            f"{filename}:",
            "FOUND" if path.is_file() else "MISSING",
            f"({path.stat().st_size:,} bytes)" if path.is_file() else ""
        )

    config_path = folder / "adapter_config.json"
    if config_path.is_file():
        with open(config_path, "r") as f:
            adapter_config = json.load(f)
        print("LoRA rank:", adapter_config.get("r"))
        print("Base model:", adapter_config.get("base_model_name_or_path"))



--- Original adapter ---
Folder exists: True
adapter_config.json: FOUND (1,103 bytes)
adapter_model.safetensors: FOUND (282,648 bytes)
LoRA rank: 1
Base model: Qwen/Qwen2.5-0.5B-Instruct

--- Unlearned adapter ---
Folder exists: True
adapter_config.json: FOUND (1,103 bytes)
adapter_model.safetensors: FOUND (282,648 bytes)
LoRA rank: 1
Base model: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

config_path = ROOT / "relearning_pilot_config.json"
rows_path = ROOT / "relearning_pilot_rows.json"

with open(config_path, "r") as f:
    config = json.load(f)

with open(rows_path, "r") as f:
    rows = json.load(f)

print("=== Relearning configuration ===")
print(json.dumps(config, indent=2))

print("\nNumber of relearning examples:", len(rows))

questions = [
    row["question"]
    for row in rows
    if isinstance(row, dict) and row.get("question")
]

print("Unique questions:", len(set(questions)))
print("Example fields:", list(rows[0].keys()) if rows else "No rows found")


=== Relearning configuration ===
{
  "seed": 20261024,
  "num_relearning_examples": 8,
  "num_steps": 8,
  "learning_rate": 1e-05,
  "lora_rank": 1,
  "lora_alpha": 1,
  "max_length": 512,
  "evaluation_forget_examples": 8,
  "evaluation_retain_examples": 100,
  "note": "Feasibility pilot only. Use identical relearning rows and training settings for unlearned and fresh-model conditions. Do not train on held-out evaluation examples."
}

Number of relearning examples: 8
Unique questions: 8
Example fields: ['question', 'answer']


In [ ]:

import torch
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))
else:
    print("No GPU detected. Do not start training yet.")

for package in ["transformers", "peft", "datasets", "accelerate", "safetensors"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56
transformers: 5.20.2
peft: 0.21.2
datasets: 4.8.5
accelerate: 1.15.0
safetensors: 0.8.0


In [ ]:

%pip install -U "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:

import importlib.metadata as metadata
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PEFT:", metadata.version("peft"))
print("torchao:", metadata.version("torchao"))


PyTorch: 2.11.0+cu130
CUDA available: True
PEFT: 0.21.2
torchao: 0.18.0


In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = (
    "/content/drive/MyDrive/qwen_tofu_research/"
    "checkpoints/unlearned_lora_r1_pilot"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False,
)

model.eval()

print("Embedding device:", model.get_input_embeddings().weight.device)
print("First parameter device:", next(model.parameters()).device)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

assert model.get_input_embeddings().weight.device.type == "cuda"
assert next(model.parameters()).device.type == "cuda"
assert not any(p.requires_grad for p in model.parameters())

print("\nSUCCESS: Adapter loaded; GPU checks passed; no training performed.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Embedding device: cuda:0
First parameter device: cuda:0
Trainable parameters: 0

SUCCESS: Adapter loaded; GPU checks passed; no training performed.


In [ ]:
import gc
import json
import random
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

UNLEARNED_ADAPTER = ROOT / "checkpoints" / "unlearned_lora_r1_pilot"
OUT_UNLEARNED = ROOT / "checkpoints" / "relearned_unlearned_lora_r1_pilot"
OUT_FRESH = ROOT / "checkpoints" / "relearned_fresh_lora_r1_pilot"

with open(ROOT / "relearning_pilot_rows.json", encoding="utf-8") as f:
    rows = json.load(f)

with open(ROOT / "relearning_pilot_config.json", encoding="utf-8") as f:
    cfg = json.load(f)

assert len(rows) == 8
assert not OUT_UNLEARNED.exists(), (
    f"Output already exists: {OUT_UNLEARNED}. "
    "Choose a new output directory rather than overwriting it."
)
assert not OUT_FRESH.exists(), (
    f"Output already exists: {OUT_FRESH}. "
    "Choose a new output directory rather than overwriting it."
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_example(row):
    question = row["question"]
    answer = row.get("answer", row.get("reference"))

    if not answer:
        raise ValueError("Missing reference answer.")

    messages = [{"role": "user", "content": question}]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages + [{"role": "assistant", "content": answer}],
        tokenize=False,
        add_generation_prompt=False,
    )

    # Character boundary is used to mask prompt tokens.
    common_prefix = 0
    for a, b in zip(prompt_text, full_text):
        if a != b:
            break
        common_prefix += 1

    if common_prefix != len(prompt_text):
        raise ValueError("Prompt is not a prefix of full conversation.")

    enc = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=cfg["max_length"],
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    input_ids = enc["input_ids"]
    offsets = enc["offset_mapping"][0]
    labels = input_ids.clone()

    for i, (start, end) in enumerate(offsets.tolist()):
        if end <= common_prefix:
            labels[0, i] = -100

    # At least one answer token must be available for loss.
    if int((labels[:, 1:] != -100).sum()) == 0:
        raise ValueError("No answer tokens left after masking.")

    return {
        "input_ids": input_ids,
        "attention_mask": enc["attention_mask"],
        "labels": labels,
    }

train_batches = [build_training_example(r) for r in rows]
print("Prepared training examples:", len(train_batches))

def make_base_model():
    return AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": 0},
    )


def train_condition(condition, output_dir):
    # Same seed, data order, and optimization budget in each condition.
    random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg["seed"])

    print(f"\nStarting condition: {condition}")
    base = make_base_model()

    if condition == "unlearned":
        model = PeftModel.from_pretrained(
            base,
            str(UNLEARNED_ADAPTER),
            is_trainable=True,
        )
    elif condition == "fresh":
        lora_cfg = LoraConfig(
            r=cfg["lora_rank"],
            lora_alpha=cfg["lora_alpha"],
            lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(base, lora_cfg)
    else:
        raise ValueError(condition)

    # Explicitly place the complete model on the T4 GPU.
    model = model.to("cuda:0")

    model.train()
    model.config.use_cache = False

    embedding_device = model.get_input_embeddings().weight.device
    parameter_devices = {p.device for p in model.parameters()}

    print("Embedding device:", embedding_device)
    print("Parameter devices:", parameter_devices)

    assert embedding_device.type == "cuda", (
        f"Embedding is on {embedding_device}, not CUDA"
    )
    assert parameter_devices == {torch.device("cuda:0")}, (
        f"Model parameters are on unexpected devices: {parameter_devices}"
    )

    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    print("Trainable parameters:", trainable)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg["learning_rate"],
    )

    logs = []
    for step in range(cfg["num_steps"]):
        # Exactly one example per step; same order in both conditions.
        batch = train_batches[step % len(train_batches)]
        batch = {k: v.to("cuda") for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        result = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        loss = result.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            1.0,
        )
        optimizer.step()

        value = float(loss.detach().float().cpu())
        logs.append({"step": step + 1, "loss": value})
        print(f"{condition} step {step + 1}/{cfg['num_steps']}: loss={value:.4f}")

        del batch, result, loss

    output_dir.mkdir(parents=True, exist_ok=False)
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    with open(output_dir / "training_log.json", "w", encoding="utf-8") as f:
        json.dump({
            "condition": condition,
            "config": cfg,
            "logs": logs,
            "trainable_parameters": trainable,
        }, f, indent=2)

    print("Saved adapter to:", output_dir)

    del optimizer, model, base
    gc.collect()
    torch.cuda.empty_cache()

Prepared training examples: 8


In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Training examples:", len(train_batches))
print("Unlearned adapter exists:", UNLEARNED_ADAPTER.is_dir())
print("Unlearned output exists:", OUT_UNLEARNED.exists())
print("Fresh output exists:", OUT_FRESH.exists())

CUDA available: True
Training examples: 8
Unlearned adapter exists: True
Unlearned output exists: False
Fresh output exists: False


In [ ]:
train_condition("unlearned", OUT_UNLEARNED)


Starting condition: unlearned


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Embedding device: cuda:0
Parameter devices: {device(type='cuda', index=0)}
Trainable parameters: 67584
unlearned step 1/8: loss=1.8618
unlearned step 2/8: loss=1.4852
unlearned step 3/8: loss=2.7009
unlearned step 4/8: loss=2.0169
unlearned step 5/8: loss=1.3854
unlearned step 6/8: loss=2.3244
unlearned step 7/8: loss=2.8715


unlearned step 8/8: loss=2.0589
Saved adapter to: /content/drive/MyDrive/qwen_tofu_research/checkpoints/relearned_unlearned_lora_r1_pilot


In [ ]:
train_condition("fresh", OUT_FRESH)


Starting condition: fresh


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Embedding device: cuda:0
Parameter devices: {device(type='cuda', index=0)}
Trainable parameters: 67584
fresh step 1/8: loss=1.9193
fresh step 2/8: loss=1.5198
fresh step 3/8: loss=2.7583
fresh step 4/8: loss=2.0486
fresh step 5/8: loss=1.4927
fresh step 6/8: loss=2.3425
fresh step 7/8: loss=2.9468
fresh step 8/8: loss=2.0797
Saved adapter to: /content/drive/MyDrive/qwen_tofu_research/checkpoints/relearned_fresh_lora_r1_pilot


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

output_dirs = {
    "Unlearned → relearned": (
        ROOT / "checkpoints" / "relearned_unlearned_lora_r1_pilot"
    ),
    "Fresh model → relearned": (
        ROOT / "checkpoints" / "relearned_fresh_lora_r1_pilot"
    ),
}

for label, folder in output_dirs.items():
    print(f"\n--- {label} ---")
    print("Folder exists:", folder.is_dir())

    for filename in ["adapter_config.json", "adapter_model.safetensors",
                     "training_log.json"]:
        path = folder / filename
        print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")

    log_path = folder / "training_log.json"
    if log_path.is_file():
        with open(log_path, "r", encoding="utf-8") as f:
            log = json.load(f)
        losses = [entry["loss"] for entry in log["logs"]]
        print("Completed steps:", len(losses))
        print("Initial loss:", round(losses[0], 4))
        print("Final loss:", round(losses[-1], 4))




--- Unlearned → relearned ---
Folder exists: True
adapter_config.json: FOUND
adapter_model.safetensors: FOUND
training_log.json: FOUND
Completed steps: 8
Initial loss: 1.8618
Final loss: 2.0589

--- Fresh model → relearned ---
Folder exists: True
adapter_config.json: FOUND
adapter_model.safetensors: FOUND
training_log.json: FOUND
Completed steps: 8
Initial loss: 1.9193
Final loss: 2.0797


In [ ]:
import json
import gc
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

UNLEARNED_ADAPTER = ROOT / "checkpoints" / "unlearned_lora_r1_pilot"
RELEARNED_UNLEARNED = ROOT / "checkpoints" / "relearned_unlearned_lora_r1_pilot"
RELEARNED_FRESH = ROOT / "checkpoints" / "relearned_fresh_lora_r1_pilot"

# Existing held-out questions and reference answers
with open(ROOT / "experiment_splits.json", "r", encoding="utf-8") as f:
    splits = json.load(f)

print("Split file keys:", list(splits.keys()))
print("CUDA available:", torch.cuda.is_available())
print("All three adapter folders exist:", all(
    p.is_dir() for p in [
        UNLEARNED_ADAPTER, RELEARNED_UNLEARNED, RELEARNED_FRESH
    ]
))

Split file keys: ['seed', 'pilot_train_questions', 'forget_train_rows', 'forget_eval_rows', 'retain_eval_rows']
CUDA available: True
All three adapter folders exist: True


In [ ]:
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")

with open(ROOT / "experiment_splits.json", "r", encoding="utf-8") as f:
    splits = json.load(f)

for split_name in ["forget_eval_rows", "retain_eval_rows"]:
    rows = splits[split_name]
    print(f"\n--- {split_name} ---")
    print("Number of examples:", len(rows))
    if rows:
        print("First example type:", type(rows[0]).__name__)
        print("First example:", json.dumps(
            rows[0], ensure_ascii=False, indent=2
        )[:2500])


--- forget_eval_rows ---
Number of examples: 8
First example type: dict
First example: {
  "question": "Which awards has Nikolai Abilov won for his contribution to literature?",
  "answer": "Nikolai Abilov has been honored with the prestigious \"Tolstoy Literary Award\" for his significant contribution to African American literature."
}

--- retain_eval_rows ---
Number of examples: 100
First example type: dict
First example: {
  "question": "Who is the author born in Melbourne, Australia on July 26, 1956 known for their literary fiction?",
  "answer": "The author born in Melbourne, Australia on July 26, 1956 who is known for their work in literary fiction is Rory Greenfield."
}


In [ ]:
import json
import gc
import re
import string
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

ROOT = Path("/content/drive/MyDrive/qwen_tofu_research")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

CONDITIONS = {
    "unlearned": ROOT / "checkpoints" / "unlearned_lora_r1_pilot",
    "unlearned_relearned": ROOT / "checkpoints" / "relearned_unlearned_lora_r1_pilot",
    "fresh_relearned": ROOT / "checkpoints" / "relearned_fresh_lora_r1_pilot",
}

PREDICTIONS_PATH = ROOT / "relearning_eval_predictions.json"
METRICS_PATH = ROOT / "relearning_eval_metrics.json"

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
assert all(path.is_dir() for path in CONDITIONS.values()), \
    "At least one adapter folder is missing."
assert not PREDICTIONS_PATH.exists(), \
    f"{PREDICTIONS_PATH.name} already exists; don't overwrite it."
assert not METRICS_PATH.exists(), \
    f"{METRICS_PATH.name} already exists; don't overwrite it."

with open(ROOT / "experiment_splits.json", "r", encoding="utf-8") as f:
    splits = json.load(f)

forget_rows = splits["forget_eval_rows"]
retain_rows = splits["retain_eval_rows"]
eval_rows = (
    [{"split": "forget", **r} for r in forget_rows]
    + [{"split": "retain", **r} for r in retain_rows]
)

assert len(forget_rows) == 8 and len(retain_rows) == 100
assert all("question" in r and "answer" in r for r in eval_rows)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def normalize_answer(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(prediction, reference):
    return float(normalize_answer(prediction) == normalize_answer(reference))

def rouge_l_f1(prediction, reference):
    pred = normalize_answer(prediction).split()
    ref = normalize_answer(reference).split()
    if not pred or not ref:
        return 0.0

    # Longest common subsequence, using a memory-efficient row.
    previous = [0] * (len(ref) + 1)
    for token in pred:
        current = [0]
        for j, ref_token in enumerate(ref, start=1):
            if token == ref_token:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    lcs = previous[-1]
    precision = lcs / len(pred)
    recall = lcs / len(ref)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def load_condition(adapter_path):
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": 0},
    )
    model = PeftModel.from_pretrained(
        base, str(adapter_path), is_trainable=False
    )
    model.eval()
    model.config.use_cache = True

    assert model.get_input_embeddings().weight.device.type == "cuda"
    return model

def make_prompt(question):
    messages = [
        {"role": "system", "content": "Answer the question as accurately as possible."},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

@torch.inference_mode()
def evaluate_model(model, rows):
    records = []

    for idx, row in enumerate(rows, start=1):
        prompt = make_prompt(row["question"])
        prompt_ids = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=512
        )["input_ids"].to("cuda:0")

        reference = row["answer"]
        reference_ids = tokenizer(
            reference, return_tensors="pt", add_special_tokens=False,
            truncation=True, max_length=256
        )["input_ids"].to("cuda:0")

        # Generate an answer from the question.
        generated = model.generate(
            input_ids=prompt_ids,
            attention_mask=torch.ones_like(prompt_ids),
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated_ids = generated[0, prompt_ids.shape[1]:]
        prediction = tokenizer.decode(
            generated_ids, skip_special_tokens=True
        ).strip()

        # NLL of the reference answer conditioned on the question prompt.
        # The prompt tokens are masked out of the loss.
        full_ids = torch.cat([prompt_ids, reference_ids], dim=1)
        attention_mask = torch.ones_like(full_ids)
        labels = full_ids.clone()
        labels[:, :prompt_ids.shape[1]] = -100

        output = model(
            input_ids=full_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        target_token_count = reference_ids.shape[1]
        nll = float(output.loss.item())

        records.append({
            "split": row["split"],
            "question": row["question"],
            "reference": reference,
            "prediction": prediction,
            "exact_match": exact_match(prediction, reference),
            "rouge_l": rouge_l_f1(prediction, reference),
            "reference_nll": nll,
            "reference_tokens": int(target_token_count),
        })

        del prompt_ids, reference_ids, generated, full_ids
        del attention_mask, labels, output

        if idx % 20 == 0 or idx == len(rows):
            print(f"  Evaluated {idx}/{len(rows)} examples")

    return records

all_predictions = {}

for condition_name, adapter_path in CONDITIONS.items():
    print(f"\nEvaluating: {condition_name}")
    model = load_condition(adapter_path)
    all_predictions[condition_name] = evaluate_model(model, eval_rows)

    del model
    gc.collect()
    torch.cuda.empty_cache()

def summarize(records, split_name):
    subset = [r for r in records if r["split"] == split_name]
    return {
        "n": len(subset),
        "exact_match": sum(r["exact_match"] for r in subset) / len(subset),
        "rouge_l": sum(r["rouge_l"] for r in subset) / len(subset),
        "mean_reference_nll": sum(r["reference_nll"] for r in subset) / len(subset),
    }

metrics = {
    condition: {
        split: summarize(records, split)
        for split in ["forget", "retain"]
    }
    for condition, records in all_predictions.items()
}

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, ensure_ascii=False, indent=2)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\nSaved predictions:", PREDICTIONS_PATH)
print("Saved metrics:", METRICS_PATH)

for condition, split_metrics in metrics.items():
    print(f"\n--- {condition} ---")
    for split, values in split_metrics.items():
        print(
            f"{split}: n={values['n']}, "
            f"EM={values['exact_match']:.4f}, "
            f"ROUGE-L={values['rouge_l']:.4f}, "
            f"mean reference NLL={values['mean_reference_nll']:.4f}"
        )


Evaluating: unlearned


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Evaluated 20/108 examples
  Evaluated 40/108 examples
  Evaluated 60/108 examples
  Evaluated 80/108 examples
  Evaluated 100/108 examples
  Evaluated 108/108 examples

Evaluating: unlearned_relearned


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Evaluated 20/108 examples
  Evaluated 40/108 examples
  Evaluated 60/108 examples
  Evaluated 80/108 examples
  Evaluated 100/108 examples
  Evaluated 108/108 examples

Evaluating: fresh_relearned


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Evaluated 20/108 examples
  Evaluated 40/108 examples
  Evaluated 60/108 examples
  Evaluated 80/108 examples
  Evaluated 100/108 examples
  Evaluated 108/108 examples

Saved predictions: /content/drive/MyDrive/qwen_tofu_research/relearning_eval_predictions.json
Saved metrics: /content/drive/MyDrive/qwen_tofu_research/relearning_eval_metrics.json

--- unlearned ---
forget: n=8, EM=0.0000, ROUGE-L=0.1613, mean reference NLL=1.6656
retain: n=100, EM=0.0000, ROUGE-L=0.1890, mean reference NLL=2.2804

--- unlearned_relearned ---
forget: n=8, EM=0.0000, ROUGE-L=0.1630, mean reference NLL=1.6650
retain: n=100, EM=0.0000, ROUGE-L=0.1897, mean reference NLL=2.2797

--- fresh_relearned ---
forget: n=8, EM=0.0000, ROUGE-L=0.1592, mean reference NLL=1.6724
retain: n=100, EM=0.0000, ROUGE-L=0.1765, mean reference NLL=2.2916
